<div style="text-align:center;">
  <img src="https://github.com/LinkedEarth/Logos/blob/master/PaleoPAL/PaleoPal_rectangular_light.png?raw=true" width="500">
</div>

## PaleoPAL Evaluation: Notebook 2

This notebook is part of a series of evaluation tests for the [PaleoPAL](linked.earth/paleopal) assistant. You have two hours to complete the assignment. 

The notebook is divided into the following sections:
1. Data Gathering (1 hour and 15min)
2. Analysis (30 min)
3. Visualization (15 min)

If you cannot complete the assignment for each section in the time alloted, use the solution and move on to the next section. 

**Use VS Code to complete the assignment**. 

In [50]:
#### Import libraries

import io

import matplotlib.pyplot as plt
import pandas as pd
import requests
from scipy.interpolate import CubicSpline

import pyleoclim as pyleo
import pyleotups as pt
from pylipd.lipd import LiPD

## Motivation

Age models provide the relationship between depth and time in ice core records, but they are often revised as new methods, data, or calibrations become available. As a result, the same ice core record may appear in multiple databases with different age models. In this exercise, you will search for the Vostok ice core record in one database (the LiPDGraph), identify its existing age model, and update it using a more recent or alternative age model available from another database (PANGAEA). Your ultimate goal is to update the LiPDGraph with the latest age model.

## Data Gathering (1 hour and 15 min)

The task can be summarized as follows: first, identify the Vostok $\delta D$ record in LiPDGraph and determine whether it spans the past 400 kyr. If it does, you should download the data as LiPD as the format will make it easier to re-upload the updated dataset to the graph. If it does not, you will need to create your own LiPD file based on the PANGAEA record, which is more time consumming. Working in LiPD format will . The revised age model that you will use for this update is currently hosted by PANGAEA. 

The following three sections walk you through the data processing. 

## PyleoTUPS

Let's start the task by obtaining the new age model from [Bouchet et al. (2023)](https://doi.pangaea.de/10.1594/PANGAEA.990385). In this dataset, the new age model is `ice_age_2023`. 

**Assignment task:** 
a. Search for the study using the information on from the [PANGAEA page](https://doi.pangaea.de/10.1594/PANGAEA.990385)
b. Identify the columns corresponding to the depth in the archive and the new age model
c. Place the depth and age information in a new `pandas.DataFrame`.

In [49]:
ds = pt.PangaeaDataset()
df_data = ds.get_data(study_id='990385')[0]

df = pd.concat([df_data['Depth ice/snow'], df_data['Ice age_2']], axis=1)

df.head()

## LiPDGraph

Using a SPARQL query on the LiPDGraph, look for datasets that could correspond to the Vostok record and assess if they cover the last 400kyr. 

**Assignment task:**
a. Query the LiPDGraph endpoint (https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic) and look for the Vostok record. There are many ways to do so, from looking for name Vostok in the name of the dataset or the site name or restricting geographical coordinates to get as close as possible to the site. Return the necessary information to complete part b.  

b. Assess whether the record(s) in the LiPDGraph cover the past 400kyr and retrieve relevant timeseries information (such as name of the variables, units, values...). 

In [48]:
url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'

query = """PREFIX le: <http://linked.earth/ontology#>
PREFIX le_var: <http://linked.earth/ontology/paleo_variables#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?dataSetName ?paleoData_variableName ?time_variableName ?time_units ?archiveType ?minTime ?maxTime ?geo_siteName
WHERE {
  ?ds le:hasName ?dataSetName .

  ?ds le:hasPaleoData ?data .
  ?data le:hasMeasurementTable ?table .

  ?table le:hasVariable ?var .
  ?var le:hasName ?paleoData_variableName .
  FILTER(?paleoData_variableName != "age" && ?paleoData_variableName != "year")

  ?table le:hasVariable ?timevar .
  ?timevar le:hasName ?time_variableName .
  OPTIONAL { ?timevar le:hasUnits ?tuObj . ?tuObj rdfs:label ?time_units . }
  FILTER(?time_variableName = "age")

  OPTIONAL{?timevar le:hasMinValue ?minTime .}
  OPTIONAL{?timevar le:hasMaxValue ?maxTime .}

  ?ds le:hasLocation ?loc .
  ?loc le:hasSiteName ?geo_siteName .
  FILTER(REGEX(?geo_siteName, "vostok", "i"))

  ?ds le:hasArchiveType ?archiveTypeObj .
  ?archiveTypeObj rdfs:label ?archiveType .
  FILTER(REGEX(?archiveType, "glacier", "i"))

}
"""

response = requests.post(url, data = {'query': query})

data = io.StringIO(response.text)
df_graph = pd.read_csv(data, sep=",")
df_graph

### PyLiPD

In addition to the LiPDGraph, you have access to a LiPD file made a few years ago that contains the older age model also found on PANGAEA. Using PyLiPD, load the data in `Vostok.Bazin.2013.lpd` and retrieve timeseries information about the $\delta D$ timeseries. 

**Assignment Task:**

a. Using PyLiPD, open the dataset and retrieve relevant timeseries information. 

b. Filter the resulting dataframe to keep only rows where the paleo variable is $\delta D$

In [47]:
lipd = LiPD()
lipd.load('Vostok.Bazin.2013.lpd')

df_l = lipd.get_timeseries_essentials(lipd.get_all_dataset_names())
df_l

## Analysis (30 min)

Age models are constructed by establishing a relationship between depth in the archive and time. There are many methods for doing this. In this exercise, you will use a cubic spline interpolation. Once you have generated the new age model, you will compare the original and updated versions to evaluate how changes in the age model affect the $\delta D$ record.

**Assignment task:**

a. Fit a cubic spline between depth in the archive and the new age model from the Bouchet et al. (2023) dataset. This will define the relationship between depth and “new age.” Several toolboxes can be used to perform spline interpolation, including `scipy`.

b. Apply the spline model to the Bazin et al. (2013) data.

c. Create two `pyleoclim.Series` objects: one using the original age model reported in Bazin et al. (2013), and the other using the new spline-interpolated age model.


In [46]:
depth2age2023 = CubicSpline(df['Depth ice/snow'], df['Ice age_2'], extrapolate=True)
age_2023 = depth2age2023(df_l['depth_values'].iloc[0])

ts_2013 = pyleo.Series(time = df_l['time_values'].iloc[0],
                       value = df_l['paleoData_values'].iloc[0],
                       time_unit = df_l['time_units'].iloc[0],
                       value_unit = df_l['paleoData_units'].iloc[0],
                       time_name = df_l['time_variableName'].iloc[0],
                        value_name = df_l['paleoData_variableName'].iloc[0],
                        verbose=False)

ts_2023 = pyleo.Series(time = age_2023,
                       value = df_l['paleoData_values'].iloc[0],
                       time_unit = df_l['time_units'].iloc[0],
                       value_unit = df_l['paleoData_units'].iloc[0],
                       time_name = df_l['time_variableName'].iloc[0],
                        value_name = df_l['paleoData_variableName'].iloc[0],
                        verbose=False)

## Visualization (15 min)

**Assignment task:**

Create a single figure with two side-by-side panels to compare the results. In the first panel, plot the two $\delta D$ time series together. This will allow you to visually assess how the updated age model affects the timing of variability in the Vostok record. In the second panel, create a scatter plot comparing the original Bazin et al. (2013) ages against the updated ages obtained from the Bouchet et al. (2023) age model. This comparison will help you identify where the two age models agree and where they diverge.

In [45]:
ms = pyleo.MultipleSeries([ts_2013, ts_2023])


fig,ax  = plt.subplots(1, 2, figsize=(24,6))
ms.plot(ax=ax[0], title='Vostok $\delta D$')
ax[1].scatter(ts_2013.time, ts_2023.time)
ax[1].set_xlabel('2013 Age (ka)')
ax[1].set_ylabel('2023 Age (ka)')